# Bài 27 — chuỗi luyện Easy · LeetCode #136 — Single Number

**Mức độ:** Easy · **Ngôn ngữ:** Python 3 · **Đề gốc:** https://leetcode.com/problems/single-number/

**Mục tiêu học tập**

- Đọc đề và tự phân biệt input/output, constraints và ví dụ biên.
- Nắm cú pháp Python theo từng cell thực nghiệm trước khi ghép thuật toán.
- Hiểu brute force/baseline trước, sau đó giải thích mấu chốt tối ưu bằng bất biến.
- Trace từng bước với dữ liệu dễ sai, không xem code như một hộp đen.
- Chạy lời giải chuẩn LeetCode, test mẫu, edge case, randomized/exhaustive khi phù hợp.
- Tự giải **6 bài luyện từ dễ đến khó**, chỉ xem gợi ý/đáp án cuối sau khi thử.

**Cách học:** đọc Markdown → tự dự đoán output → chạy cell Python ngay sau đó → giải thích lại bằng lời. Các cell TODO không chạy tự động và chỉ phục vụ tự luyện.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Mảng `nums` có đúng **một** số xuất hiện một lần; **tất cả số khác xuất hiện đúng hai lần**. Hãy tìm số đơn lẻ, yêu cầu thời gian tuyến tính và bộ nhớ phụ hằng số.

### Ví dụ quan trọng

`[2,2,1] → 1`; `[4,1,2,1,2] → 4`; `[1] → 1`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import List, Optional
from collections import deque, Counter

### 2.1. Số nguyên và biểu diễn nhị phân

Mọi số nguyên có thể hình dung là một dãy **bit** 0 và 1.
Ví dụ 5 có nhị phân `101`, 3 là `011` nếu cùng nhìn trong 3 bit.
Trong Python `bin(5)` in `'0b101'`: tiền tố `0b` báo cách viết nhị phân.
Trong bài này, ta không cần tự chuyển qua string bit để tính; Python có sẵn toán tử bitwise.
Hiểu từng bit giúp lý giải XOR chứ không chỉ học thuộc dòng `ans ^= x`.
`n` là độ dài mảng; số bit của giá trị là một khái niệm khác.

In [ ]:
for number in (0,1,2,3,5,7,12):
    print(number,'->',bin(number), 'định dạng 4 bit:',format(number,'04b'))
assert int('101',2)==5

### 2.2. Bảng chân trị XOR từng bit

**XOR (exclusive OR)** cho bit 1 khi **hai bit khác nhau**, bit 0 khi giống nhau.
Bảng bốn dòng: `0^0=0`, `0^1=1`, `1^0=1`, `1^1=0`.
Ở Python, viết `^` để XOR; dấu `**` lại là lũy thừa, hoàn toàn khác nhau.
Ví dụ `5^3`: `101^011=110`, kết quả 6.
Phải phân biệt XOR của số nguyên theo bit với câu hỏi True/False về cả số.
`^` và `!=` không thể thay thế nhau trong thuật toán này.

In [ ]:
for a in (0,1):
    for b in (0,1):
        print(f'{a} ^ {b} = {a^b}')
print('5 (101) XOR 3 (011) =',5^3,'=',format(5^3,'03b'))
assert (5^3)==6

### 2.3. Vì sao `a ^ a = 0` và `a ^ 0 = a`?

XOR so từng bit độc lập. Mỗi bit của một số luôn bằng chính nó,
nên hai bản sao ở cùng bit cho `0`. Do vậy `a ^ a == 0`, kể cả a là số âm.
Bit 0 khác bit a chỉ khi bit của a là 1, nên `a ^ 0 == a`.
Đây là **hai quy tắc triệt tiêu** quyết định thuật toán. Một số xuất hiện 2 lần
sẽ biến mất khỏi phép XOR tổng, còn số đơn lẻ không có bản sao để biến mất.
Hãy chạy với cả số 0, số dương và số âm trước khi suy luận với cả mảng.

In [ ]:
for a in (0,1,5,99,-7):
    print('a=',a,'a^a=',a^a,'a^0=',a^0)
    assert a^a==0 and a^0==a

### 2.4. Tính giao hoán và kết hợp — đổi thứ tự không đổi kết quả

XOR vừa **giao hoán** `a^b == b^a`, vừa **kết hợp** `(a^b)^c == a^(b^c)`.
Nhờ đó ta có thể tưởng tượng gom hai số trùng vào cạnh nhau dù mảng ban đầu đan xen.
Ví dụ `[4,1,2,1,2]` tương đương `4^(1^1)^(2^2)` và bằng 4.
Mảng không cần sort; XOR triệt tiêu cặp *bất kể vị trí*.
Nhưng mọi số khác phải xuất hiện **đúng hai lần**, nếu không chứng minh này có thể sai.
Thử vài bộ ba để tự quan sát cả hai tính chất.

In [ ]:
for a,b,c in ((3,5,7),(0,8,8),(-2,9,4)):
    print((a,b,c),'giao hoán?',a^b==b^a,'kết hợp?',(a^b)^c==a^(b^c))
    assert (a^b)==(b^a) and ((a^b)^c)==(a^(b^c))

### 2.5. `^=` là phép cập nhật tích lũy, không phải so sánh

`answer ^= x` chính là `answer = answer ^ x`.
Nếu bắt đầu `answer=0`, mỗi bước lưu XOR của **toàn bộ tiền tố đã duyệt**.
Đây là *accumulator* (biến tích lũy), không phải số lần xuất hiện.
Với `[2,2,1]`: `0^2=2`, `2^2=0`, `0^1=1`.
Nhìn rõ trước/sau cập nhật giúp bạn dùng bất biến khi chứng minh đúng.
Giá trị tạm có thể lên xuống, không nhất thiết đơn điệu.

In [ ]:
acc=0
for x in [2,2,1]:
    before=acc
    acc ^= x
    print('x=',x,'acc trước=',before,'acc sau=',acc)
assert acc==1

### 2.6. `dict`, `Counter` và cách nghĩ dễ hiểu trước khi tối ưu

Cách tự nhiên là đếm số lần mỗi giá trị xuất hiện: `counts[x] += 1`.
`Counter(nums)` từ `collections` tạo bảng **giá trị -> tần suất**.
Với `[4,1,2,1,2]`, đếm `{4:1,1:2,2:2}` rồi tìm key có count 1.
Phương án này rất trực quan nhưng lưu tới `O(n)` giá trị khác nhau;
trong khi đề yêu cầu bộ nhớ bổ sung không tăng theo n.
Vì vậy XOR là kỹ thuật mới cần học, không có lý do cắt bỏ bước đếm tần suất.

In [ ]:
from collections import Counter
data=[4,1,2,1,2]
freq=Counter(data)
print('tần suất:',dict(freq))
print('giá trị xuất hiện đúng 1 lần:',[x for x,c in freq.items() if c==1])
assert freq[4]==1 and freq[1]==2

### 2.7. Số âm, số 0 và số nguyên kích thước cố định

Python hỗ trợ XOR với **số âm**: `-3 ^ -3 == 0`, `-3 ^ 0 == -3`.
Không cần tự dựng mảng 32 bit cho bài này. LeetCode giới hạn giá trị
trong miền số nguyên hữu hạn; vì vậy accumulator được xem là dùng `O(1)` extra space.
Nếu mở rộng bài toán sang số nguyên dài tùy ý, chi phí theo số bit `b` cũng cần xét:
thực hiện XOR trên các số b-bit không còn là chi phí hằng độc lập với b.
Điều kiện của đề là **một phần tử đơn lẻ** và mọi phần tử khác xuất hiện đôi;
nếu 3 lần hoặc 4 lần thì phải phân tích lại phép XOR, không áp dụng máy móc.

In [ ]:
for data in ([0,1,1],[-3,5,5],[-4,-4,-8],[0,0,1]):
    acc=0
    for x in data:
        acc ^= x
    print(data,'->',acc)
assert (-3^-3)==0 and (-3^0)==-3

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Toán tử XOR `^` so sánh bit: khác nhau → 1, giống nhau → 0. Tính chất: `a^a=0`, `a^0=a`, XOR có thể đổi thứ tự nhóm.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
for a,b in [(2,2),(2,0),(4,1),(1,2)]:
    print(a,'^',b,'=',a^b,'| nhị phân',bin(a),'^',bin(b),'=',bin(a^b))

### Câu hỏi gợi mở trước khi tối ưu

Sau khi tự chạy ví dụ nền tảng, hãy tự trả lời bằng lời:

1. Trong bài **Single Number**, đâu là dữ liệu thật, đâu là công cụ dựng thử?
2. Biến nào thay đổi sau một vòng lặp / sau một lời gọi đệ quy?
3. Khi nào một ký tự / node / giá trị được xem là 'đã xử lý xong'?
4. Nếu input rỗng, có một phần tử, hay chỉ có một nhánh, cần return kiểu gì?
5. Thao tác nào tăng theo số phần tử n, thao tác nào chỉ giữ vài biến?

Không cần trả lời bằng công thức ngay; thử input nhỏ rồi diễn đạt kết quả.

### Quy trình chạy notebook không phụ thuộc trạng thái ngầm

Các hàm dựng input, baseline, `solve`, trace và đáp án được định nghĩa theo thứ tự từ trên xuống dưới.
Không chạy phần đáp án trước phần bài tập nếu muốn tự luyện.
Khi thử một biến thể, hãy tạo input mới, không sửa input đã được dùng cho `assert` ở các cell phía trên.
Nếu chạy lại một cell, bạn nên kiểm tra các biến demo bị ghi đè; cách ổn định nhất là **Restart Kernel & Run All**.

Ở các bài Linked List/Tree, LeetCode đã cung cấp sẵn lớp node. Trong notebook, lớp node được định nghĩa để mô phỏng local và không phải mã cần nộp.

### 3.1. Thử XOR bằng từng bit thay vì chỉ nhìn đáp số thập phân

Tưởng tượng ba số được viết thành các hàng nhị phân có cùng độ rộng.
Khi XOR, mỗi **cột bit** hoạt động độc lập: số lượng bit 1 chẵn thì tổng XOR của cột bằng 0,
số lượng bit 1 lẻ thì bằng 1. Đây chính là tính chất phép XOR gộp.
Ví dụ `4^1^2^1^2`: những cột từ số 1 và 2 xuất hiện đôi nên đều bị triệt tiêu,
chỉ còn các bit từ số 4. Chúng ta có thể in accumulator bằng `format(x,'04b')`.
Đó là bước nối từ toán học sang lệnh Python `ans ^= x`.

In [ ]:
data136=[4,1,2,1,2]
ans=0
for x in data136:
    before=ans
    ans ^= x
    print(f'{before:04b} ^ {x:04b} = {ans:04b} (số {ans})')
assert ans==4

### 3.2. Bảng đếm và bảng XOR nhìn hai mặt của cùng dữ liệu

Bảng tần suất trả lời câu hỏi **mỗi số xuất hiện bao nhiêu lần**,
còn bảng XOR chỉ giữ phép gộp đã triệt tiêu các cặp.
Baseline dùng dictionary cần nhiều bộ nhớ để lưu thông tin từng số;
thuật toán XOR chỉ cần **một biến kết quả** vì điều kiện bài toán đã cung cấp
đủ cấu trúc để bỏ qua thông tin tần suất chi tiết.
Nếu không có điều kiện mọi số còn lại xuất hiện đúng hai lần,
một accumulator là chưa đủ để xác định phần tử đơn lẻ.
So sánh cách tính hai bảng trên một input mẫu.

In [ ]:
from collections import Counter
sample136=[-2,11,-2,8,8]
freq=Counter(sample136)
x_result=0
for number in sample136:x_result ^= number
print('bảng đếm:',dict(freq))
print('kết quả XOR:',x_result)
assert [n for n,c in freq.items() if c==1]==[x_result]

## 4. Cách nghĩ trực giác (baseline)

Dùng `dict` đếm số lần xuất hiện, rồi lấy số có số lần = 1. Đúng nhưng tốn O(n) bộ nhớ phụ.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(nums):
    counts={}
    for x in nums:counts[x]=counts.get(x,0)+1
    for x,count in counts.items():
        if count==1:return x

In [ ]:
for a in [[2,2,1],[4,1,2,1,2],[-3,7,-3]]:print(a,baseline(a))

### 4.1. Baseline dùng Counter đúng nhưng tốn bộ nhớ

Cách dễ nhất là đếm rồi tìm tần suất 1: đúng vì đề khẳng định
chỉ một số xuất hiện một lần. Với mảng độ dài n,
dictionary/Counter có tối đa xấp xỉ `(n+1)/2` key nếu mọi số khác xuất hiện đôi.
Vì vậy time trung bình `O(n)`, extra space `O(n)`.
Đề còn yêu cầu giải pháp **constant extra space**, nên dù baseline đã nhanh,
nó chưa đạt toàn bộ điều kiện của bài. Đây là ví dụ tuyệt vời về bài toán
đã có `O(n)` time nhưng vẫn còn động cơ tối ưu memory.

In [ ]:
from collections import Counter
for sample in ([2,2,1],[4,1,2,1,2],[-9,1,-9]):
    table=Counter(sample)
    single=next(x for x,count in table.items() if count==1)
    print(sample,'->',dict(table),'-> số lẻ:',single)
    assert single==baseline(sample)

### 4.2. Vì sao sort không phải hướng tốt khi đề yêu cầu O(n) time?

Nếu sắp xếp, các số xuất hiện đôi sẽ đứng cạnh nhau;
ta có thể đi từng cặp và tìm số bị lạc ra.
Tuy nhiên sort thông thường cần `O(n log n)` thời gian,
lâu hơn yêu cầu `O(n)` của bài. `sorted(nums)` còn tạo list mới độ dài n.
Có một khác biệt quan trọng giữa **ý tưởng đúng** và **đáp ứng constraints**.
Dùng phép XOR vừa không sort, vừa không tạo bảng đếm.
Hãy quan sát ảnh hưởng của sắp xếp lên dữ liệu; đây chỉ là minh họa, không phải lời giải để nộp.

In [ ]:
samples=[[4,1,2,1,2],[-1,7,7,5,5],[8,8,9]]
for sample in samples:
    sorted_copy=sorted(sample)
    print('gốc:',sample,'sorted mới:',sorted_copy)
    assert sample!=[] and sorted_copy==sorted(sample)

## 5. Vì sao cần chuyển sang kỹ thuật chính?

XOR tự triệt tiêu các cặp: `x^x=0`. Vì mọi số khác xuất hiện đúng hai lần, XOR **tất cả** phần tử sẽ chỉ còn số xuất hiện một lần.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

### Bước đột phá — Tần suất đôi được mã hóa bằng phép XOR

Bảng `Counter` giữ mọi key để biết số nào có tần suất 1. Ta tự hỏi:
**Có một phép gộp nào khiến các số xuất hiện đôi tự mất đi không?**
XOR đáp ứng đúng: `x ^ x = 0`, `0 ^ y = y`, và ta có thể đổi vị trí / nhóm hạng tử.
Do đó `a ^ a ^ b ^ b ^ unique = unique` dù các phần tử ở thứ tự nào.

**Bất biến:** sau khi xử lý k phần tử, `answer` bằng XOR của đúng k phần tử đầu.
Khởi tạo 0 thoả mãn khi k=0. Mỗi bước `answer ^= nums[k]`
chính xác nối thêm phần tử mới. Sau n phần tử, các cặp bị triệt tiêu,
đáp án còn lại là số xuất hiện đúng một lần.

**Không tổng quát hóa sai:** nếu số khác xuất hiện 3 lần, XOR ba bản sao bằng chính số đó.
Vì vậy chứng minh phụ thuộc điều kiện xuất hiện **hai lần**.

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Giữ `result=0`; với mỗi `x`, cập nhật `result ^= x`. Kết quả cuối là số đơn lẻ. Không cần đếm hoặc sắp xếp.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Khi nào giữ lại số, khi nào triệt tiêu?

Khi duyệt mảng, chúng ta không cố quyết định một số nào đã xuất hiện hay chưa.
XOR tự triệt tiêu nếu số đó có bản sao ở bất cứ vị trí nào sau này.
Vì vậy accumulator tạm thời có thể chưa phải đáp án; ví dụ `[4,1,2,1,2]`
lần lượt ra 4,5,7,6,4. Chỉ khi hoàn tất mảng mới chắc chắn còn số đơn lẻ.
Bài này khác hẳn Two Sum dùng `seen`: không cần tra cứu từng key.
`ans ^= num` không lấy tối đa, không đếm số lần; nó đổi các bit theo parity.

In [ ]:
trace_nums136=[4,1,2,1,2]
steps=[];ans=0
for i,num in enumerate(trace_nums136):
    before=ans;ans ^= num
    steps.append((i,num,before,ans))
for step in steps:print('index, num, ans trước, ans sau =',step)
assert ans==4

In [ ]:
def solve(nums: list[int]) -> int:
    result=0
    for x in nums:
        result^=x
    return result

### 6.1. Viết hàm `solve` với một accumulator duy nhất

Lời giải có ba bước chính: đặt `result=0`, duyệt từng x trong nums,
rồi `result ^= x`, cuối cùng `return result`.
Không cần `sort`, `Counter`, `set`, lẫn hai vòng lặp.
Hàm nộp LeetCode dùng cú pháp `def singleNumber(self, nums: List[int]) -> int:`;
trong notebook, type hint `list[int]` cũng chạy tốt với Python 3 hiện đại.
Hãy thử cả mảng chứa 0 và số âm, sau đó tự nói bất biến bằng lời.
Giá trị chính xác không phụ thuộc thứ tự input.

In [ ]:
for values in ([2,2,1],[4,1,2,1,2],[0],[0,1,1],[-5,-5,-99]):
    answer=solve(values)
    print(values,'=>',answer)
    assert answer==baseline(values)

### 6.2. XOR bằng `functools.reduce`: hiểu rồi mới dùng

`reduce(function, iterable, initial)` gộp dãy bằng một phép toán hai ngôi.
Với XOR, `reduce(xor, nums, 0)` cho cùng kết quả với vòng for đã học.
Đây **không phải** cách mặc định để dạy người mới vì ký hiệu trừu tượng hơn.
Nó hữu ích để hiểu thuật toán đang làm một phép **fold** trên dãy.
Dùng vòng for trong code nộp thường dễ đọc và trace hơn.
Mỗi `reduce` vẫn cần duyệt n phần tử; không làm thời gian tốt hơn O(n).

In [ ]:
from functools import reduce
from operator import xor
for data in ([4,1,2,1,2],[2,2,1],[-7,3,3]):
    folded=reduce(xor,data,0)
    print(data,'reduce XOR:',folded,'vòng for:',solve(data))
    assert folded==solve(data)

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| x đang đọc | result trước | Phép XOR | result sau |
|---:|---:|---|---:|
| 4 | 0 | `0^4` | 4 |
| 1 | 4 | `4^1` | 5 |
| 2 | 5 | `5^2` | 7 |
| 1 | 7 | `7^1` | 6 |
| 2 | 6 | `6^2` | **4** |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [ ]:
def trace_xor(nums):
    cur=0
    for i,x in enumerate(nums):
        print('i',i,'x',x,'trước',cur, '->',cur^x)
        cur^=x
    return cur
print('kết quả:',trace_xor([4,1,2,1,2]))

### 7.1. Trace bảng với dữ liệu chứa số đan xen

Bảng có bốn cột: chỉ số i, phần tử x, `ans` trước XOR, `ans` sau XOR.
Với `[4,1,2,1,2]`, các giá trị tạm là 4 → 5 → 7 → 6 → 4.
Từng giá trị tạm không phải số đơn lẻ chắc chắn, trừ khi đã đọc hết dãy.
Khi gặp bản sao của 1, những bit do 1 đóng góp được đổi lại;
tới bản sao của 2, toàn bộ đóng góp của 2 cũng biến mất.
Viết lại bảng bằng tay, sau đó chạy hàm bên dưới để tự chấm.

In [ ]:
def trace_xor136(nums):
    answer=0
    print('i | x | ans trước | ans sau | bit sau')
    for i,x in enumerate(nums):
        old=answer;answer ^= x
        print(f'{i:>1} | {x:>2} | {old:>9} | {answer:>7} | {answer & 15:04b}')
    return answer
assert trace_xor136([4,1,2,1,2])==4

### 7.2. Dữ liệu âm vẫn triệt tiêu được như số dương

Python biểu diễn phép bitwise với số âm theo ngữ nghĩa tương ứng
với biểu diễn hai bù có độ rộng đủ lớn. Ta **không** cần ép sang nhị phân 32 bit.
Quan sát `[7,-3,7]`: kết quả -3. Cặp 7 bị triệt tiêu dù số đơn là số âm.
Nếu hiển thị `bin(-3)`, bạn thấy `'-0b11'`, không phải chuỗi hai bù vô hạn;
đây là cách **in string**, không đổi bản chất của toán tử XOR.
Hãy tập trung vào thuộc tính `a ^ a == 0`, không vào độ rộng chuỗi in ra.

In [ ]:
cases=[(7,-3,7),(-1,42,42),(0,-5,-5)]
for nums in cases:
    print('dãy:',nums,'số âm ở bin:',[bin(x) for x in nums],'đáp án:',solve(list(nums)))
    assert solve(list(nums))==baseline(list(nums))

### 7.3. Đếm parity của bit như một cách hiểu độc lập

Ta có thể tự kiểm chứng XOR bằng cách tính **số bit 1 của mỗi cột modulo 2**.
Với input gồm các số **không âm** nhỏ (để minh họa), nếu bit thứ j xuất hiện
số lần lẻ thì bit j của đáp án bằng 1; nếu số lần chẵn thì bằng 0.
Đây không phải cách nộp vì phải đi qua nhiều cột, nhưng nó giúp nhìn bản chất XOR:
phép cộng theo modulo 2 trên từng cột, không nhớ thứ tự xuất hiện.
Để kiểm tra các bit của một số, `(x >> j) & 1` lấy bit thứ j.

In [ ]:
numbers=[6,3,6,9,9]
by_bits=0
for bit in range(4):
    ones=sum((x>>bit)&1 for x in numbers)
    if ones%2:by_bits |= (1<<bit)
    print('bit',bit,'số bit 1:',ones,'parity:',ones%2)
print('dựng lại:',by_bits,'solve:',solve(numbers))
assert by_bits==solve(numbers)==3

### 7.4. Lý do điều kiện của đề quan trọng hơn mẫu ví dụ

Thử `[2,2,2,1]`: XOR cho 3, nhưng số xuất hiện 3 lần là 2,
không thoả mãn điều kiện của bài gốc, nên kết quả 3 không có ý nghĩa.
Trên dữ liệu `[5,5,5,5,7]`, XOR vẫn vô tình cho 7 vì 4 là số chẵn,
nhưng điều đó không cho phép ta phát biểu bài đúng với mọi tần suất.
Trước khi áp dụng một mẹo tối ưu, hãy trích đúng **tiền đề** để chứng minh.
Đây là kỹ năng nhận dạng bài: 'mỗi phần tử còn lại xuất hiện đúng hai lần' chính là chìa khóa.

In [ ]:
invalid=[2,2,2,1]
print('input sai ràng buộc:',invalid,'XOR=',solve(invalid))
assert solve(invalid)==3
print('Không dùng kết quả này như đáp án của bài vì input ngoài constraints.')

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Sau khi duyệt i phần tử, `result` bằng XOR của đúng i phần tử đó. XOR giao hoán và kết hợp nên các số xuất hiện hai lần triệt tiêu, còn đúng một số duy nhất. Với số âm, XOR số nguyên Python vẫn thỏa `a^a=0` và `a^0=a`.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
class Solution:
    def singleNumber(self, nums: list[int]) -> int:
        result = 0
        for x in nums:
            result ^= x
        return result

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for a,expected in [([2,2,1],1),([4,1,2,1,2],4),([1],1),([-3,7,-3],7),([0,5,0],5)]:
    assert solve(a)==expected and Solution().singleNumber(a)==expected and baseline(a)==expected
print('PASS 5 ca XOR')

### 10.1. Kiểm thử sinh mảng hợp lệ ngẫu nhiên (differential)

Tạo một số đơn lẻ `unique` và một tập giá trị khác, mỗi giá trị thêm đúng **hai lần**.
Trộn toàn bộ mảng để tránh việc các cặp luôn đứng kề nhau.
Đối chiếu `solve`, `baseline` và `Solution().singleNumber`.
Ta cố tình sinh cả số âm, 0, số dương và nhiều cặp;
không sinh dữ liệu sai cấu trúc vì thuật toán không được yêu cầu xử lý ngoại lệ đó.
Khi `assert` sai, in cả mảng khiến ta dễ tái hiện lỗi.

In [ ]:
import random
rng136=random.Random(136)
for test_i in range(1200):
    unique=rng136.randrange(-100,101)
    population=[x for x in range(-100,101) if x!=unique]
    chosen=rng136.sample(population,rng136.randrange(0,9))
    nums=[unique]+[item for x in chosen for item in (x,x)]
    rng136.shuffle(nums)
    expected=baseline(nums)
    assert expected==unique
    assert solve(nums)==expected and Solution().singleNumber(nums)==expected
print('PASS 1200 mảng hợp lệ được sinh ngẫu nhiên')

### 10.2. Đổi thứ tự mảng không được đổi kết quả

XOR có tính giao hoán và kết hợp, nên mọi phép hoán vị của cùng một multiset
phải có kết quả XOR giống nhau. Đây là **thuộc tính bảo toàn** để test.
Kiểm với nhiều cách trộn cùng dữ liệu thay vì chỉ một input cố định.
Nếu một lời giải dùng sai vị trí phần tử hoặc dừng sớm theo chỉ số,
phép test này có thể phát hiện được.
Quan trọng: đây là test **bổ trợ**, không thay thế điều kiện mọi phần tử khác xuất hiện đúng 2 lần.

In [ ]:
data=[4,1,2,1,2]
expected=solve(data)
import random
rr136=random.Random(2136)
for _ in range(300):
    other=data.copy();rr136.shuffle(other)
    assert solve(other)==expected
print('PASS 300 hoán vị mảng giữ nguyên số đơn')

### 10.3. Độ dài lớn, vẫn không cấp phát bảng đếm trong solve

Một trường hợp có hàng nghìn cặp số sẽ cho baseline bảng tần suất rất lớn,
trong khi `solve` chỉ có `result` và biến `num` tại mỗi lượt.
Kiểm thử này không dùng `print` mọi bước để không kéo dài output notebook.
Giá trị số đơn được đặt ở vị trí xa đầu dãy để người đọc không nhầm rằng
thuật toán có thể kết luận ngay khi thấy số xuất hiện một lần ở đầu.
Cỡ test đủ để phát hiện vòng lặp sai, nhưng không thay cho phân tích Big-O.

In [ ]:
large_nums=list(range(3000))*2+[99999]
import random
random.Random(1360).shuffle(large_nums)
assert solve(large_nums)==99999
assert len(large_nums)==6001
print('PASS dãy dài',len(large_nums),'phần tử, kết quả',solve(large_nums))

## 11. Time / Space Complexity — trong điều kiện số nguyên của đề

Gọi `n = len(nums)`. Baseline đếm bằng Counter: Time **O(n)** trung bình, extra Space **O(n)** cho bảng tần suất.
Lời giải XOR: duyệt đúng n phần tử, mỗi lượt một XOR trên số nguyên thuộc phạm vi đề; Time **O(n)** và extra Space **O(1)**.
Không tính input là extra space. Nếu bạn mở rộng sang số nguyên có độ dài bit tùy ý b, phép XOR có chi phí theo b; ở bài LeetCode đang xét số nguyên có miền giới hạn, nên phân tích O(1) trên từng thao tác là hợp lý.
XOR không dùng sort nên tránh chi phí O(n log n).

## 12. Các lỗi người mới hay mắc

- ❌ Dùng cộng/trừ để triệt tiêu số sẽ không xử lý đúng mọi tình huống.
- ❌ Áp dụng XOR khi một số xuất hiện **3 lần**: đề không cho điều đó.
- ❌ Dùng `^` tưởng là lũy thừa; trong Python lũy thừa là `**`.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

In [ ]:
sample_bug136=[4,1,2,1,2]
print('nhìn theo bit:',[(x,bin(x)) for x in sample_bug136])
print('XOR đúng:',solve(sample_bug136),'tổng thường sai:',sum(sample_bug136))
assert solve(sample_bug136)==4

### 12.1. `^` khác `**`: lỗi cú pháp làm đổi hoàn toàn phép toán

`a ^ b` là XOR bitwise; `a ** b` là lũy thừa.
Ví dụ `2^2=0`, còn `2**2=4`. Nhầm ký tự sẽ dẫn đến kết quả sai.
Nếu code chạy mà ra những số cực lớn hoặc không triệt tiêu cặp,
hãy in thử bảng chân trị với hai số đơn giản.
Không thể sửa bằng cách thêm điều kiện if riêng cho vài giá trị;
phải sửa đúng **phép toán**. Đây là lỗi phổ biến khi chuyển từ ký hiệu toán học
`x²` sang code Python.

In [ ]:
print('2^2:',2^2,'2**2:',2**2)
print('5^5:',5^5,'5**5:',5**5)
assert (2^2)==0 and (2**2)==4

### 12.2. Dùng sum thay XOR không hề triệt tiêu cặp

Trong cộng thông thường, hai bản sao x cho `2*x`, không biến mất.
Vì vậy `sum(nums)` không thể là số đơn lẻ.
Một công thức khác `2*sum(set(nums))-sum(nums)` có thể tìm số đơn nếu
các phần tử còn lại xuất hiện hai lần, nhưng tốn `O(n)` bộ nhớ cho set.
Nó không đáp ứng yêu cầu extra space `O(1)` của bài.
Điều kiện tối ưu là phần của **đề**, không phải sở thích: cần đọc cả thời gian và bộ nhớ.

In [ ]:
demo=[4,1,2,1,2]
print('sum sai:',sum(demo))
print('set-formula đúng nhưng O(n) nhớ:',2*sum(set(demo))-sum(demo))
print('XOR O(1) extra space:',solve(demo))
assert sum(demo)!=solve(demo)

## 13. Nhận diện dạng bài và vận dụng

Mảng có "mọi phần tử đúng 2 lần, trừ 1" và cần O(1) bộ nhớ → **XOR cancellation**.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Hoán vị ngẫu nhiên thứ tự mảng: XOR vẫn cho cùng kết quả vì có tính giao hoán.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
for a in [[4,1,2,1,2],[2,4,2,1,1],[1,2,4,2,1]]:print(a,solve(a))

### 14.1. Biến thể: tìm một số bị thiếu trong `[0, n]`

Xét mảng độ dài n gồm n số phân biệt lấy từ tập `0,1,...,n`,
thiếu đúng một số. XOR tất cả giá trị 0..n rồi XOR các giá trị có trong mảng:
mọi số có mặt đều xuất hiện hai lần trong biểu thức và bị triệt tiêu,
còn duy nhất số bị thiếu.
Đây là **bài khác** (#268), nhưng cùng invariant của XOR.
Time O(n), extra space O(1) với miền số nguyên có giới hạn.
Đừng nhầm rằng `singleNumber` của #136 có thể dùng trực tiếp cho input này
mà không XOR thêm dãy đủ 0..n.

In [ ]:
def missing_number_xor136(nums):
    answer=len(nums)
    for i,value in enumerate(nums):
        answer ^= i ^ value
    return answer
for sample in ([3,0,1],[0,1],[9,6,4,2,3,5,7,0,1]):
    print(sample,'-> số thiếu:',missing_number_xor136(sample))
assert missing_number_xor136([3,0,1])==2

### 14.2. Không dùng XOR y hệt khi mọi số khác xuất hiện ba lần

Nếu mỗi số bình thường xuất hiện **ba lần**, XOR 3 bản sao bằng chính số đó:
`x^x^x=x`. Vậy không thể áp dụng lời giải #136 nguyên xi.
Một hướng khác là đếm từng bit theo modulo 3 (ở miền số nguyên có độ rộng cố định),
hoặc dùng kỹ thuật hai mặt nạ bit cho bài Single Number II.
Mục đích biến thể này là học cách **kiểm tra giả định** của pattern trước khi dùng.
Hãy quan sát một phản ví dụ mà XOR trả sai, còn Counter vẫn xác định được số đơn;
không cần cố cài trọn bài nâng cao ở đây.

In [ ]:
triple_case=[2,2,2,7]
xor_result=solve(triple_case)
correct_result=next(x for x,c in Counter(triple_case).items() if c==1)
print('3 lần mỗi số thường:',triple_case,'XOR sai:',xor_result,'đáp án thực:',correct_result)
assert xor_result!=correct_result

## 15. Hệ thống luyện tập — 6 bài từ cơ bản đến nâng cao

**Bài 1 — Dự đoán output Python:** Dự đoán `(5 ^ 5, 5 ^ 0, 5 ^ 3)` và viết các bit giải thích vì sao.

**Bài 2 — Trace tay:** Trace `[4,1,2,1,2]`, ghi `(i,num,answer trước,answer sau)` đủ cả 5 lượt.

**Bài 3 — Hoàn thiện code chính:** tự viết hàm `exercise_single136(nums)` mà không nhìn cell nộp.

**Bài 4 — Biến thể:** Tìm số thiếu trong `[0..n]` khi input có n số phân biệt, bằng XOR của dãy đầy đủ và dãy đã cho.

**Bài 5 — Viết test phản ví dụ:** Tự tạo 5 mảng hợp lệ `(nums,expected)` với một phần tử lẻ; có 0, số âm, mảng 1 phần tử, các cặp đan xen.

**Bài 6 — Mini challenge:** Nâng cao: mảng có ĐÚNG HAI số xuất hiện một lần, số còn lại xuất hiện hai lần. Trả hai số đơn theo thứ tự tăng dần bằng XOR phân hoạch bit.

**Cách tự học:** làm bài 1–2 trên giấy → viết TODO bài 3–5 → tự kiểm bằng test của chính mình → thử bài 6. Hãy ghi rõ lý do chọn trạng thái, ranh giới vòng lặp và điều kiện dừng; **đáp án tham khảo nằm sau toàn bộ đề bài**.

### Bài 3 — TODO: tự viết lại Single Number

Viết code vào hàm ở cell tiếp theo theo đúng kiểu input/output của đề.
Không gọi hàm TODO khi chạy toàn bộ notebook. Khi hoàn thành, tự bỏ `raise NotImplementedError` rồi chạy test được gợi ý dưới hàm.
**Tiêu chí tự chấm:** giải thích được vì sao code đúng ngay cả ở trường hợp nhỏ nhất, và không làm thao tác dư thừa so với lời giải chính.

In [ ]:
def exercise_single136(nums):
    # TODO 1: tạo accumulator bằng 0
    # TODO 2: duyệt từng x trong nums
    # TODO 3: dùng phép XOR ^= và return kết quả cuối
    raise NotImplementedError("Tự viết trước khi kiểm thử")
# assert exercise_single136([4,1,2,1,2]) == 4

### Bài 4 — TODO: đổi yêu cầu, giữ kỹ thuật cốt lõi

Tìm số thiếu trong `[0..n]` khi input có n số phân biệt, bằng XOR của dãy đầy đủ và dãy đã cho.

Đừng chép y nguyên code submit: hãy chỉ ra đầu ra thay đổi ở đâu; tự tạo 2 input mới để kiểm thử.

In [ ]:
def exercise_missing136(nums):
    # TODO: với n = len(nums), xor tất cả 0..n và nums.
    # Dữ liệu nums chỉ chứa n số phân biệt trong miền 0..n.
    raise NotImplementedError("Chưa làm bài 4")
# assert exercise_missing136([3,0,1]) == 2

### Bài 5 — TODO: tạo bộ kiểm thử có chủ đích

Tự tạo 5 mảng hợp lệ `(nums,expected)` với một phần tử lẻ; có 0, số âm, mảng 1 phần tử, các cặp đan xen.

Mỗi test phải có **expected tự tính trước**, không được lấy `expected=solve(input)`. Một test tốt thường có mục tiêu bắt một kiểu sai cụ thể. Nếu có node, dựng dữ liệu mới ở mỗi lượt test.

In [ ]:
def exercise_cases136():
    # TODO: trả [(nums, expected), ...] có ít nhất 5 ca hợp lệ.
    # expected phải suy ra từ điều kiện đề, không được gọi solve để tính.
    raise NotImplementedError("Chưa làm bài 5")

### Bài 6 — Mini challenge chuyển giao kỹ thuật (khó)

**Yêu cầu:** Nâng cao: mảng có ĐÚNG HAI số xuất hiện một lần, số còn lại xuất hiện hai lần. Trả hai số đơn theo thứ tự tăng dần bằng XOR phân hoạch bit.

    Bài 6 không thuộc đề gốc. Hãy tự phân tích **miền input, output, tính đúng**
    và chọn một phản ví dụ nhỏ trước khi code. Nếu thay đổi cấu trúc lời giải,
    cần nói rõ điều kiện nào đã thay đổi so với bài gốc.
    Không gọi hàm có TODO trong lần chạy toàn bộ notebook: để nguyên `NotImplementedError`
    cho tới khi bạn tự điền mã và tự bật test.
    **Kiểm tra:** so kết quả với đáp án cuối, nhưng chỉ sau khi đã tự làm.

In [ ]:
def exercise_two_single_numbers136(nums):
    # TODO 1: XOR toàn bộ để nhận mix = a ^ b.
    # TODO 2: chọn một bit khác nhau của a và b.
    # TODO 3: chia nums thành hai nhóm theo bit ấy và XOR mỗi nhóm.
    # TODO 4: trả sorted([a,b]).
    raise NotImplementedError("Mini challenge chưa giải")
# assert exercise_two_single_numbers136([1,2,1,3,2,5]) == [3,5]

## 16. Gợi ý theo ba cấp — chỉ đọc sau khi tự thử

**Cấp 1:** hỏi tại sao một cặp giống nhau tự xóa được; nhớ `x^x=0` và `0^x=x`.
**Cấp 2:** accumulator cập nhật `answer ^= value` trong đúng một vòng `for`.
**Cấp 3:** Bài 4: XOR từ `0` đến `n`, tiếp tục XOR từng phần tử input; không cần sort.
**Nâng cao:** Với hai số đơn a,b, `mix=a^b != 0`. Bit thấp nhất bật `mask=mix & -mix` chia a,b vào hai nhóm khác nhau; cặp trùng vẫn ở cùng nhóm và triệt tiêu.

**Cách sử dụng gợi ý:** bắt đầu ở cấp 1; nếu vẫn chưa ra, chỉ mở cấp 2 rồi tự viết tiếp. Không đọc lời giải đầy đủ ngay khi vừa gặp khó. Với bài 6, thử tự xác định input nhỏ nhất gây sai trong phương án bạn đề xuất.

## 17. Đáp án tham khảo — chỉ đọc sau khi đã làm TODO

Các hàm **đáp án** dùng tên khác với `exercise_*` để không vô tình thực thi code chưa hoàn thiện.
Đáp án bài 3 được định nghĩa trước, rồi đến các biến thể bài 4–6 và bộ test tương ứng.
Sau khi chạy, hãy so sánh logic từng dòng với lời giải chính và tìm lý do các cài đặt tương đương, không chỉ so con số output.

In [ ]:
def exercise_core_answer(nums):
    result=0
    for x in nums:
        result^=x
    return result

print("Đã định nghĩa đáp án tham khảo; xem lại từng dòng rồi tự gõ lại.")

### Đáp án bài 4 — biến thể với yêu cầu đầu ra mới

Bài 4 yêu cầu: Tìm số thiếu trong `[0..n]` khi input có n số phân biệt, bằng XOR của dãy đầy đủ và dãy đã cho.

    So với lời giải gốc, hãy chỉ ra phần nào giữ nguyên:
    cú pháp duyệt, trạng thái trước/sau, bất biến. Phần thay đổi phải tương ứng
    đúng với **đầu ra mới**, không làm thay chữ ký code submit #
    hoặc vô tình lẫn thêm điều kiện ngoài phạm vi bài gốc.
    Đoạn đáp án dưới đây có assert riêng để kiểm tính đúng.

In [ ]:
def answer_missing136(nums):
    answer=len(nums)
    for i,x in enumerate(nums):
        answer ^= i ^ x
    return answer
for values,expected in [([3,0,1],2),([0,1],2),([9,6,4,2,3,5,7,0,1],8)]:
    assert answer_missing136(values)==expected
print('PASS đáp án bài 4 — số bị thiếu')

### Đáp án bài 6 và kiểm thử đáp án bài 5

**Bài 6:** Nâng cao: mảng có ĐÚNG HAI số xuất hiện một lần, số còn lại xuất hiện hai lần. Trả hai số đơn theo thứ tự tăng dần bằng XOR phân hoạch bit.

    Đọc suy luận trước khi so code: giữ trạng thái nào; tại mỗi bước
    trạng thái cập nhật ra sao; vì sao khi dừng, kết quả đáp ứng yêu cầu.
    Bài 5 là tự xây bộ test: đáp án tham khảo còn có các mẫu độc lập,
    chạy kèm assert nhưng **không hề chạy TODO của người học**.
    Nếu bộ test bạn tự nghĩ khác, điều đó bình thường: hãy đảm bảo cả
    test của bạn và test tham khảo đều có expected đúng.

In [ ]:
def answer_two_single_numbers136(nums):
    mix=0
    for x in nums:mix ^= x
    mask=mix & -mix  # lấy 1 bit mà hai số đơn khác nhau
    a=b=0
    for x in nums:
        if x & mask:a ^= x
        else:b ^= x
    return sorted([a,b])
for nums,expected in [([1,2,1,3,2,5],[3,5]),([0,-2,9,9],[ -2,0]),([-1,3,3,12],[-1,12])]:
    assert answer_two_single_numbers136(nums)==expected
print('PASS đáp án mini challenge XOR 2 số đơn')

cases136=[([1],1),([2,2,1],1),([4,1,2,1,2],4),([0,2,2],0),([-3,7,-3],7),([-9,-9,-1],-1)]
for values,expected in cases136: assert solve(values)==expected
print('PASS 6 test tham khảo bài 5')

### Lời giải thích và đáp án ngắn cho cả 6 bài

**Bài 1:** `5^5=0`, `5^0=5`, `5^3=6` (nhị phân 101 XOR 011 = 110).
**Bài 2:** dãy accumulator lần lượt 4, 5, 7, 6, 4. Chỉ giá trị cuối là đáp án chắc chắn.
**Bài 3:** `exercise_core_answer` duyệt đúng n phần tử và XOR các số.
**Bài 4:** `answer_missing136` khởi tạo bằng n rồi XOR cặp `i^nums[i]`.
**Bài 5:** có bộ test độc lập ở cell cuối; mọi input đều thỏa một số xuất hiện một lần, các số khác đúng hai lần.
**Bài 6:** `answer_two_single_numbers136` dùng bit khác biệt chia hai nhóm, vẫn O(n) time và O(1) extra space.

Đối với bài tập viết code, hãy so sánh cả **điều kiện lặp**, **thứ tự cập nhật** và **cách return** chứ không chỉ so giá trị cuối. Các hàm `answer_*` của bài 4 và 6 nằm ở những cell ngay phía trên để người học có thể chạy riêng sau khi tự làm.

## 18. Tóm tắt và checklist trước khi nộp

**Tư duy phải tự diễn đạt được bằng lời:** nêu bài toán bằng một câu, chỉ rõ trạng thái nào đủ để quyết định bước kế tiếp, rồi mới viết code. Đọc lại các ví dụ khó trong notebook; khi một điều kiện `if` làm bạn phân vân, hãy lập bảng giá trị **ngay trước và sau** câu lệnh ấy.

1. Tôi đã phân biệt đầu vào/đầu ra, và đọc đúng chữ ký `class Solution` của bài.
2. Tôi đã tự chạy thử ví dụ nhỏ, biên, phản ví dụ và kiểm thử đối chiếu nhiều cách giải.
3. Tôi hiểu từng cấu trúc Python, có thể giải thích vì sao dùng nó thay vì cách khác.
4. Tôi chỉ ra được bất biến: sau bước thứ k, điều gì **luôn đúng** và tại sao?
5. Tôi phân tích Time và Space dựa trên số lượng phần tử/nút/bít và có tính tới output/stack/chuỗi tạm.
6. Tôi đã hoàn thành TODO **trước** khi kéo tới đáp án tham khảo.
7. Tôi chỉ sao chép `class Solution` để nộp LeetCode; các hàm dựng input, trace và bộ test dành cho notebook, không dành cho submit.

**Tự đánh giá:** bạn hiểu bài khi tự viết được lời giải trên một trang trống và xử lý được một biến thể mà không sao chép mã cũ.

**Mẫu nhận dạng của bài 136:** các số thường xuất hiện đúng đôi, chỉ một số đơn → XOR tích lũy, O(n) time, O(1) extra space.